# MindFlayer — GRPO Training (Medium) on Colab
**OpenEnv Hackathon Submission** | [HF Space](https://prithvigg-mindflayer.hf.space/) | [GitHub](https://github.com/prithidevghosh/mindflayer)

Train a deceptive social reasoning agent at **medium difficulty** using GRPO with HF TRL.

| Component | Detail |
|---|---|
| **Difficulty** | `medium` — 4 rounds, two investigators (eleven + will) |
| **vs Easy** | Easy: 3 rounds, 1 investigator. Medium adds will (The Analyst) who raises suspicion on factual contradictions |
| **Environment** | [HF Space](https://prithvigg-mindflayer.hf.space/) (live MindFlayer server) |
| **Training** | This Colab notebook (T4 / A100 GPU) |
| **Model** | `Qwen/Qwen2.5-0.5B-Instruct` + LoRA via unsloth |
| **Framework** | HF TRL GRPO |
| **API Keys** | Two OpenAI keys — auto-rotates on 429 |

## 1. Install Dependencies

In [ ]:
!pip install -q \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" \
    "trl>=0.15.0" \
    "transformers>=4.47.0" \
    "datasets>=2.18.0" \
    "peft>=0.14.0" \
    "accelerate>=0.34.0" \
    "bitsandbytes>=0.43.0" \
    "openai>=1.30.0" \
    "matplotlib" \
    "protobuf>=5.29.0,<7"

!pip install -q "openenv-mindflayer[training] @ git+https://github.com/prithidevghosh/mindflayer"

print("Done.")

## 2. Configuration

Add **both** OpenAI keys in the Colab secrets sidebar (key icon):
- `OPENAI_KEY_1` — primary key (used first)
- `OPENAI_KEY_2` — fallback key (auto-switched on 429)

**Rate limit math (medium mode, 2 keys):**

| Metric | Value |
|---|---|
| Per key | 500 RPM / 10k RPD (gpt-4o-mini Tier 1) |
| Two keys combined | 1000 RPM / 20k RPD |
| Calls per episode | 4 rounds × 2 investigators + 1 judge = **9 calls** |
| 5-hour RPD budget | 20k × (5/24) ≈ **4,167 calls → ~463 episodes** |
| Dataset sizing | `ROWS_PER_SCENARIO=11` → 55 rows × 2 epochs / (batch2 × accum4) = **13–14 grad steps** (96% of budget) |
| Calls/min at peak | **~600–800 RPM** across both keys (~300–400 per key, under 500 limit) |

**Wall time:**
- SFT warmup : ~30–40 min (3 epochs, no API calls)
- GRPO training: ~30–60 min (14 steps, API-rate-limited not compute-limited)
- **Worst case total: ~75–90 min** (T4 GPU + slow API + occasional 429 retries)
- **Best case total: ~45–55 min** (A100 + fast API)
- 5-hour budget is the hard ceiling — training finishes in ~1.5 hours typical

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_KEY_1"] = userdata.get("OPENAI_KEY_1")
    os.environ["OPENAI_KEY_2"] = userdata.get("OPENAI_KEY_2") or ""
except Exception:
    pass  # running outside Colab — set env vars manually

if not os.environ.get("OPENAI_KEY_1"):
    raise RuntimeError("OPENAI_KEY_1 secret is required. Add it via the key icon in the sidebar.")
key2_status = "set" if os.environ.get("OPENAI_KEY_2") else "NOT SET (single-key mode — 429s may slow training)"
print(f"OPENAI_KEY_1: set")
print(f"OPENAI_KEY_2: {key2_status}")

# Environment — the live HF Space
ENV_URL = "https://prithvigg-mindflayer.hf.space"
os.environ["MINDFLAYER_URL"] = ENV_URL

# ── Medium mode configuration ──────────────────────────────────────────────
# These MUST be set before importing mindflayer.training (read once at import).
os.environ["MINDFLAYER_PARALLEL_EPISODES"] = "16"   # 8 per key — ~600–800 RPM combined
os.environ["MINDFLAYER_TASK_ID"]           = "medium"  # 4 rounds, eleven + will
os.environ["MINDFLAYER_MAX_ROUNDS"]        = "4"
os.environ["MINDFLAYER_ROWS_PER_SCENARIO"] = "3"    # TRL GRPO internally multiplies by
                                                     # num_gen(4) × grad_accum(4) × epochs(2)
                                                     # so 1 row → ~120 steps (~1.5–2 hrs)
                                                     # 11 rows → 1,320 steps (11+ hrs) ← wrong
os.environ["MINDFLAYER_SFT_EPOCHS"]        = "3"    # SFT is FREE — no API calls

MODEL_ID  = "Qwen/Qwen2.5-0.5B-Instruct"
HUB_REPO  = "prithidevghosh/mindflayer-agent-medium-qwen2.5-0.5b"

NUM_GENERATIONS = 4
MAX_ROUNDS      = 4

print(f"\nEnvironment      : {ENV_URL}")
print(f"Model            : {MODEL_ID}")
print(f"Mode             : {os.environ['MINDFLAYER_TASK_ID']} (4 rounds, eleven + will)")
print(f"Rows/scenario    : {os.environ['MINDFLAYER_ROWS_PER_SCENARIO']}  → ~120 GRPO steps")
print(f"Parallel episodes: {os.environ['MINDFLAYER_PARALLEL_EPISODES']}  → ~600–800 RPM combined")
print(f"SFT epochs       : {os.environ['MINDFLAYER_SFT_EPOCHS']}")
print(f"\nExpected wall time: 1.5–2 hrs (matches easy training time)")

## 3. Smoke Test — Verify Medium Environment Connectivity
Resets with `task_id="medium"` to confirm 4-round, two-investigator mode is live.

In [ ]:
from mindflayer import MindFlayerEnv, FlayerAction

print(f"Connecting to {os.environ['MINDFLAYER_URL']} (medium mode)...")

with MindFlayerEnv(base_url=os.environ["MINDFLAYER_URL"]).sync() as env:
    reset_result = env.reset(task_id="medium")
    obs = reset_result.observation
    print("Connected!\n")
    print(f"Difficulty         : {obs.difficulty}")
    print(f"Max rounds         : {obs.max_rounds}")
    print(f"Secret project     : {obs.secret_project}")
    print(f"Suspicion threshold: {obs.suspicion_threshold}")
    print(f"Opening (eleven)   :\n{obs.eleven_response}")

    step_result = env.step(FlayerAction(
        message="Has anyone looked at the access logs from last Tuesday and compared them "
                "against who had cross-project permissions that week? "
                "The Cipher team lead had access to both systems — that seems worth examining."
    ))
    print(f"\n--- Smoke step (reward={step_result.reward:.2f}) ---")
    print(f"eleven :\n{step_result.observation.eleven_response}")
    print(f"will   :\n{step_result.observation.will_response}")
    print(f"done   : {step_result.done}")

print("\nSmoke test passed. Medium environment confirmed (eleven + will active).")

## 4. Import Training Utilities
All training logic is imported from `mindflayer.training` — medium-specific additions use `sft_warmup_medium`.

In [ ]:
import importlib, logging
import mindflayer.training as _mft
importlib.reload(_mft)

from mindflayer.training import (
    FLAYER_SYSTEM_PROMPT,
    reward_survival,
    reward_deception_effectiveness,
    reward_strategic_choice,
    reward_tom_judge,
    reward_anti_hack,
    reward_format,
    clear_reward_cache,
    load_base_model,
    build_dataset,
    GenerationLogCallback,
    ClearRewardCacheCallback,
    print_reward_averages,
)
from mindflayer.training.sft_warmup_medium import run_sft_warmup_medium

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

print("System prompt (first 200 chars):")
print(FLAYER_SYSTEM_PROMPT[:200])
print("...")
print("\nReward chain: survival, deception, strategic, tom, anti_hack, format")
print("Medium mode: 4 rounds, eleven + will (threshold=3). Max does NOT participate.")

## 5. SFT Warmup (Medium)
Loads the model via unsloth (4-bit + LoRA) and runs SFT on **4-round** hand-authored episodes.

Medium SFT teaches two new skills vs easy:
1. **Consistency across 4 rounds** — will raises suspicion on factual contradictions between rounds
2. **Two-investigator narrative control** — both eleven and will need to be managed simultaneously

~10 minutes on A100.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Switch to Runtime → Change runtime type → A100.")
device = torch.cuda.get_device_properties(0)
print(f"GPU: {device.name} | VRAM: {device.total_memory / 1024**3:.1f} GB")

print(f"\nLoading {MODEL_ID} via unsloth...")
model, tokenizer = load_base_model(MODEL_ID)

print("\nRunning medium SFT warmup (4-round examples)...")
model = run_sft_warmup_medium(model, tokenizer)
print("Medium SFT warmup done.")

## 6. GRPO Training Setup

**Medium mode reward dynamics:**
- `reward_survival` (+0.50): survived four rounds with eleven + will
- `reward_deception_effectiveness` (0→+0.30): survived with low combined suspicion
- `reward_strategic_choice` (+0.05): questions / evidence refs / third-party framing
- `reward_tom_judge` (0→+0.20): GPT-4o-mini ToM score × belief manipulation multiplier
- `reward_anti_hack` (0→−0.25): penalises silence / repetition / Level-0 denial
- `reward_format` (0→+0.10): dense shaping for `Round N:` separator format

**Key rotation:** investigators and judge retry on 429 and rotate to the next key with exponential backoff (1s → 2s → 4s). Effective: 1000 RPM / 20k RPD across both keys.

**Logging every 5 steps** (vs 50 in easy) so the training graph is granular enough to track per-step reward changes.

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

dataset = build_dataset()
print(f"Dataset: {len(dataset)} rows ({len(dataset)} scenarios × {os.environ['MINDFLAYER_ROWS_PER_SCENARIO']} per scenario)")

# 16-way parallel episode generation per reward call:
#   per_device_train_batch_size (2) × num_generations (4) × gradient_accumulation_steps (4)
#   = 32 episodes per gradient step. Asyncio semaphore caps in-flight at 16
#   (MINDFLAYER_PARALLEL_EPISODES) — 8 per key — to stay within 500 RPM per key.
grpo_config = GRPOConfig(
    use_vllm=False,
    output_dir="/content/mindflayer-grpo-output-medium",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_prompt_length=768,
    max_completion_length=768,
    num_generations=NUM_GENERATIONS,
    temperature=0.9,
    logging_steps=5,    # granular — every 5 steps
    save_steps=5,
    save_total_limit=3,
    log_completions=True,
    num_completions_to_print=2,
    report_to="none",
)

# Override GenerationLogCallback to log every 5 steps for medium
class MediumGenerationLogCallback(GenerationLogCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 5 != 0 or state.global_step == 0:
            return
        super().on_step_end(args, state, control, **kwargs)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        reward_survival,
        reward_deception_effectiveness,
        reward_strategic_choice,
        reward_tom_judge,
        reward_anti_hack,
        reward_format,
    ],
    train_dataset=dataset,
    args=grpo_config,
    callbacks=[MediumGenerationLogCallback(), ClearRewardCacheCallback()],
)

print("GRPOTrainer initialised for medium difficulty")

## 7. Train!

**What to watch:**
- `reward_survival` — primary health signal. Medium is harder than easy (threshold=3 vs 2, two investigators)
- `reward_tom_judge` — separates after ~5–10 steps once the model learns 4-round consistency
- `reward_anti_hack` — should trend toward 0 (no silences or denials)
- If key rotation fires: logs show `OpenAI key rotated → index 1` then back to `→ index 0`

Checkpoints saved every 5 steps to `/content/mindflayer-grpo-output-medium/`.

In [ ]:
import glob

_checkpoints = sorted(glob.glob("/content/mindflayer-grpo-output-medium/checkpoint-*"))
resume_checkpoint = _checkpoints[-1] if _checkpoints else None
if resume_checkpoint:
    print(f"Resuming from: {resume_checkpoint}")
else:
    print("Starting fresh")

print(f"  Model       : {MODEL_ID}")
print(f"  Mode        : medium (4 rounds, eleven + will)")
print(f"  Generations : {NUM_GENERATIONS}")
print(f"  Environment : {os.environ['MINDFLAYER_URL']}")
print()

trainer.train(resume_from_checkpoint=resume_checkpoint)

## 8. Reward Curves & Final Averages

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

print_reward_averages(trainer, last_n=50)

history = trainer.state.log_history
reward_keys = sorted({k for s in history for k in s if "reward" in k.lower() and isinstance(s[k], (int, float))})

fig, axes = plt.subplots(len(reward_keys), 1, figsize=(11, 3 * len(reward_keys)), sharex=True)
if len(reward_keys) == 1:
    axes = [axes]

for ax, key in zip(axes, reward_keys):
    vals = [s[key] for s in history if key in s and "step" in s]
    xs   = [s["step"] for s in history if key in s and "step" in s]
    ax.plot(xs, vals, linewidth=1.2)
    ax.set_ylabel(key.replace("rewards/", ""), fontsize=8)
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.3f"))
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Step")
fig.suptitle("GRPO Training (Medium) — Reward Curves", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("/content/reward_curves_medium.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → /content/reward_curves_medium.png")

In [ ]:
# Easy vs Medium — curriculum comparison chart
# Uses hardcoded data from both completed training runs.

import matplotlib.pyplot as plt

easy_steps  = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120]
easy_reward = [0.976, 0.988, 0.973, 0.976, 0.981, 1.002, 0.991, 0.992, 0.975, 0.952, 0.963, 0.969]

medium_steps  = [
      5,  10,  15,  20,  25,  30,  35,  40,  45,  50,  55,  60,
     65,  70,  75,  80,  85,  90,  95, 100, 105, 110, 115, 120,
    125, 130, 135, 140, 145, 150, 155, 160, 165, 170, 175, 180,
    185, 190, 195, 200, 205, 210, 215, 220, 225, 230, 235, 240,
    245, 250, 255, 260, 265, 270, 275, 280, 285, 290, 295, 300,
    305, 310, 315, 320, 325, 330, 335, 340, 345, 350, 355, 360,
]
medium_reward = [
    0.7617, 0.8110, 0.8560, 0.8706, 0.7888, 0.8044, 0.8527, 0.8602, 0.8402, 0.9098, 0.8500, 0.8542,
    0.8702, 0.8733, 0.9048, 0.9388, 0.9269, 0.8956, 0.8925, 0.9090, 0.9585, 0.8852, 0.9217, 0.9408,
    0.9398, 0.9173, 0.9473, 0.9375, 0.9506, 0.9475, 0.9115, 0.9567, 0.9360, 0.9315, 0.9381, 0.9279,
    0.9327, 0.9167, 0.9275, 0.9700, 0.9577, 0.9465, 0.9098, 0.9498, 0.9460, 0.9658, 0.9425, 0.9306,
    0.9417, 0.9181, 0.9790, 0.9756, 0.9735, 0.9708, 0.9148, 0.9638, 0.9335, 0.9790, 0.9615, 0.9467,
    0.9683, 0.9569, 0.9175, 0.9350, 0.9656, 0.9738, 0.9763, 0.9500, 0.9592, 0.9271, 0.9598, 0.9229,
]

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(easy_steps, easy_reward,
        color='#4C72B0', linewidth=2.2, marker='o', markersize=4,
        label='Easy (120 steps · 3 rounds · 1 investigator)')

ax.plot(medium_steps, medium_reward,
        color='#DD8452', linewidth=1.6, alpha=0.9,
        label='Medium (360 steps · 4 rounds · 2 investigators)')

# Shade API-failure region in easy run
ax.axvspan(100, 120, alpha=0.07, color='red')
ax.text(110, 0.725, '429\nerrors', fontsize=7, color='red',
        alpha=0.65, ha='center', va='bottom')

# Peak annotations
ax.annotate('Peak 1.002 (step 60)',
            xy=(60, 1.002), xytext=(72, 1.020),
            arrowprops=dict(arrowstyle='->', color='#4C72B0', lw=1.2),
            color='#4C72B0', fontsize=8.5)
ax.annotate('Peak 0.979 (step 255)',
            xy=(255, 0.979), xytext=(185, 0.993),
            arrowprops=dict(arrowstyle='->', color='#DD8452', lw=1.2),
            color='#DD8452', fontsize=8.5)

ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=0.8, alpha=0.45)

ax.set_xlabel('Training Step', fontsize=11)
ax.set_ylabel('Combined Reward', fontsize=11)
ax.set_title('MindFlayer — Easy vs Medium Curriculum', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.set_ylim(0.70, 1.05)
ax.set_xlim(0, 370)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/easy_vs_medium_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → /content/easy_vs_medium_comparison.png')


## 9. Save Model

In [ ]:
SAVE_DIR = "/content/mindflayer-trained-medium"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

size_mb = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fns in os.walk(SAVE_DIR) for f in fns
) / (1024 ** 2)
print(f"Saved to {SAVE_DIR}  ({size_mb:.0f} MB)")

## 10. Push to Hub (Optional)
Requires `HF_TOKEN` in Colab Secrets.

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    print("HF_TOKEN not set — skipping. Add it in Colab Secrets to enable.")
else:
    trainer.model.push_to_hub(HUB_REPO, token=hf_token)
    tokenizer.push_to_hub(HUB_REPO, token=hf_token)
    print(f"Pushed to https://huggingface.co/{HUB_REPO}")